In [6]:
"""
2-phase Ensemble 
"""
#paths:
from pathlib import Path
#handling data:
import numpy as np
import pandas as pd
from yaml import load
from yaml import CLoader as Loader
from datetime import datetime
#abil functions:
from abil.tune import ModelTuner as tune
from abil.predict import ModelPredictor as predict
from abil.post import AbilPostProcessor as post

In [7]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "environment.yml").exists():
            return path
    raise FileNotFoundError("Could not find project root containing environment.yml")


PROJECT_ROOT = find_project_root()
conffile = PROJECT_ROOT / "2-phase example" / "2-phase.yml"

# Load model configuration
with conffile.open('r') as f:
    model_config = load(f, Loader=Loader)

model_config['root'] = str(PROJECT_ROOT) + '/'
model_config['local_root'] = str(PROJECT_ROOT) + '/'


In [8]:
# Load training data
targets = pd.read_csv(PROJECT_ROOT / model_config['targets'])
d = pd.read_csv(PROJECT_ROOT / model_config['training'])

# Define target
target = targets['Target'][0]

# Define predictors based on YAML
predictors = model_config['predictors']
d = d.dropna(subset=predictors)

# Split training data into X_train and y
y = d[target]
X_train = d[predictors]

print("finished loading data")


finished loading data


In [4]:

#setup model:
m = tune(X_train, y, model_config)

#run model:
m.train(model="rf")
m.train(model="xgb")
m.train(model="knn")


beginning init
length of y: 34612
training regressor
{'regressor__estimator__n_estimators': [100], 'regressor__estimator__max_features': [0.8], 'regressor__estimator__max_depth': [51], 'regressor__estimator__min_samples_leaf': [1], 'regressor__estimator__max_samples': [0.9]}
Fitting 10 folds for each of 1 candidates, totalling 10 fits
[CV 2/10] END regressor__estimator__max_depth=51, regressor__estimator__max_features=0.8, regressor__estimator__max_samples=0.9, regressor__estimator__min_samples_leaf=1, regressor__estimator__n_estimators=100;, score=(train=0.904, test=0.474) total time=   3.2s
[CV 6/10] END regressor__estimator__max_depth=51, regressor__estimator__max_features=0.8, regressor__estimator__max_samples=0.9, regressor__estimator__min_samples_leaf=1, regressor__estimator__n_estimators=100;, score=(train=0.904, test=0.442) total time=   3.2s
[CV 5/10] END regressor__estimator__max_depth=51, regressor__estimator__max_features=0.8, regressor__estimator__max_samples=0.9, regresso

[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.


[CV] END  MAE: (test=-35621.517) R2: (test=0.428) RMSE: (test=-57586.510) total time=   3.2s
[CV] END  MAE: (test=-36539.679) R2: (test=0.463) RMSE: (test=-56192.349) total time=   3.2s
[CV] END  MAE: (test=-36123.557) R2: (test=0.500) RMSE: (test=-54523.104) total time=   3.3s
[CV] END  MAE: (test=-35527.870) R2: (test=0.488) RMSE: (test=-53489.825) total time=   3.3s


[Parallel(n_jobs=12)]: Done   3 out of  10 | elapsed:    3.3s remaining:    7.6s


[CV] END  MAE: (test=-34466.168) R2: (test=0.474) RMSE: (test=-52681.502) total time=   3.5s
[CV] END  MAE: (test=-37060.992) R2: (test=0.442) RMSE: (test=-59249.070) total time=   3.5s
[CV] END  MAE: (test=-36677.555) R2: (test=0.408) RMSE: (test=-57084.029) total time=   3.6s


[Parallel(n_jobs=12)]: Done   7 out of  10 | elapsed:    3.6s remaining:    1.5s


... (36 outputs left out)

[CV] END  MAE: (test=-14664.233) R2: (test=0.572) RMSE: (test=-35619.196) total time=   4.4s
[CV] END  MAE: (test=-15573.212) R2: (test=0.556) RMSE: (test=-36737.222) total time=   4.7s


[Parallel(n_jobs=12)]: Done   7 out of  10 | elapsed:    4.8s remaining:    2.0s


[CV] END  MAE: (test=-15916.639) R2: (test=0.534) RMSE: (test=-38243.274) total time=   5.1s
[CV] END  MAE: (test=-15627.345) R2: (test=0.559) RMSE: (test=-38895.112) total time=   5.5s
[CV] END  MAE: (test=-15713.551) R2: (test=0.560) RMSE: (test=-38870.276) total time=   5.5s
exported scoring to: /home/mv23682/Documents/Abil_tutorial/ModelOutput/pro_pac/scoring/knn/Prochlorococcus_zir.sav
zir rRMSE: 154%
zir rMAE: 64%
zir R2: 0.56
execution time: 0.0 seconds


[Parallel(n_jobs=12)]: Done  10 out of  10 | elapsed:    5.6s finished


In [9]:
# Load prediction data
X_predict = pd.read_csv(PROJECT_ROOT / model_config['prediction'])
X_predict.set_index(["time", "depth", "lat", "lon"], inplace=True)
X_predict = X_predict.dropna(subset=predictors)
X_predict = X_predict[predictors]

# Setup model
m = predict(X_train, y, X_predict, model_config)

# Predict model
m.make_prediction()


initialized prediction
number of models in ensemble: 3
predicting zero-inflated regressor


finished exporting summary stats to: /home/mv23682/Documents/Abil_tutorial/ModelOutput/pro_pac/predictions/rf/clf/Prochlorococcus.nc
finished exporting summary stats to: /home/mv23682/Documents/Abil_tutorial/ModelOutput/pro_pac/predictions/rf/reg/Prochlorococcus.nc
finished exporting summary stats to: /home/mv23682/Documents/Abil_tutorial/ModelOutput/pro_pac/predictions/rf/Prochlorococcus.nc
predicting zero-inflated regressor


KeyboardInterrupt: 

In [ ]:

# Posts
targets = np.array([target])
current_date = datetime.today().strftime('%Y-%m-%d')

def do_post_poc(statistic):
    #Abundance output
    m = post(X_train, y, X_predict, model_config, statistic, datatype="abundance")
    m.export_ds(current_date)

    #POC output
    m = post(X_train, y, X_predict, model_config, statistic, datatype="poc")
    m.estimate_carbon("pg poc")
    vol_conversion = 1e3 # to convert from pg C L-1 to pg C m-3
    magnitude_conversion = 1e-24 # convert from pg C to Tg C
    integ = m.integration(m, vol_conversion=vol_conversion,
                          magnitude_conversion=magnitude_conversion)
    integ.integrated_totals(targets)
    m.export_ds(current_date)

do_post_poc(statistic="mean")
do_post_poc(statistic="ci95_UL")
do_post_poc(statistic="ci95_LL")

In [ ]:

def do_post_pic(statistic):
    #Abundance output
    m = post(X_train, y, X_predict, model_config, statistic, datatype="abundance")
    m.export_ds(current_date)

    #PIC output
    m = post(X_train, y, X_predict, model_config, statistic, datatype="pic")
    m.estimate_carbon("pg pic")
    vol_conversion = 1e3 # to convert from pg C L-1 to pg C m-3
    magnitude_conversion = 1e-24 # convert from pg C to Tg C
    integ = m.integration(m, vol_conversion=vol_conversion,
                          magnitude_conversion=magnitude_conversion)
    integ.integrated_totals(targets)
    m.export_ds(current_date)

do_post_pic(statistic="mean")
do_post_pic(statistic="ci95_UL")
do_post_pic(statistic="ci95_LL")